# Looking at basal and luminal A tiles

In [10]:
import os, zipfile
from collections import Counter
from PIL import Image
import pandas as pd
import random
import matplotlib.pyplot as plt

In [7]:
data_dir = "/data/horse/ws/mala059b-rna2wsi/data/TCGA/"
df_dir = "/data/horse/ws/mala059b-rna2wsi/data//brca_subtypes.csv"

In [ ]:
# Read subtypes and find a ZIP for one LumA and one Basal patient (try multiple LumA barcodes if needed)
subtypes = pd.read_csv(df_dir)
def barcodes_for(name):
    df = subtypes[subtypes['Majority_Subtype_mRNA'].str.lower() == name.lower()]
    if df.empty:
        return []
    return df['bcr_patient_barcode'].astype(str).tolist()
luma_barcodes = barcodes_for('LumA')
basal_barcodes = barcodes_for('Basal')
print('Found', len(luma_barcodes), 'LumA barcodes and', len(basal_barcodes), 'Basal barcodes in the CSV')
zip_files = [f for f in os.listdir(data_dir) if f.lower().endswith('.zip')]
def find_zip_for_barcodes(barcodes):
    for bc in barcodes:
        for f in zip_files:
            if bc in f or bc.replace('-', '') in f or bc[:12] in f:
                return bc, os.path.join(data_dir, f)
    return None, None
luma_barcode, luma_zip = find_zip_for_barcodes(luma_barcodes)
basal_barcode, basal_zip = find_zip_for_barcodes(basal_barcodes)
print('Chosen LumA barcode:', luma_barcode, '->', luma_zip)
print('Chosen Basal barcode:', basal_barcode, '->', basal_zip)
# report sizes if found
for name, path in [('LumA', luma_zip), ('Basal', basal_zip)]:
    if path:
        z = zipfile.ZipFile(path)
        print(f'{name} zip file found:', path, 'with', len(z.namelist()), 'files. First:', z.infolist()[0].filename)
    else:
        print(f'No zip found for {name}')

In [15]:
# open the zip files, choose 9 of the tiles randomly and display them in a 3x3 grid
def display_random_tiles(zip_path, title):
    z = zipfile.ZipFile(zip_path)
    tile_names = z.namelist()
    selected_tiles = random.sample(tile_names, 9)
    
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(title)
    
    for ax, tile_name in zip(axes.flatten(), selected_tiles):
        with z.open(tile_name) as tile_file:
            img = Image.open(tile_file)
            ax.imshow(img)
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if luma_zip:
    display_random_tiles(luma_zip, 'Randomly Chosen Tiles from LumA Patient')
if basal_zip:
    display_random_tiles(basal_zip, 'Randomly Chosen Tiles from Basal Patient')
